# Lab 2.2 – Graph-Augmented RAG for DFD Connectivity

**Goal:** Detect structural trust-boundary issues in a sample DFD and attach the governing policy controls from the Week 1 corpus.

In [ ]:
import sys
from pathlib import Path

LAB_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
WEEK1 = LAB_ROOT.parent / "01-chunking"
sys.path.insert(0, str(LAB_ROOT))
sys.path.insert(0, str(WEEK1))

from src.graph_schema import load_dfd_json
from src.graph_store import DfdGraphStore
from src.graph_rag import GraphRAG
from src.chunking import process_directory

In [ ]:
doc = load_dfd_json(LAB_ROOT / "data" / "sample_dfd.json")
g = DfdGraphStore()
g.ingest(doc)
print(g.summary())

In [ ]:
paths = g.find_paths(
    source_type="ExternalEntity",
    target_type="DataStore",
    require_crosses_trust_boundary=True,
)
print(f"Found {len(paths)} path(s) that cross a trust boundary:\n")
for p in paths:
    print(" ", p)

In [ ]:
corpus = process_directory(WEEK1 / "data")
added = g.link_controls_from_corpus(corpus)
print(f"Heuristic control links added: {added}")
print("Controls on Partner API node:", g.control_links.get("ee_partner"))

In [ ]:
# Optional: attach Week 1 hybrid retriever for policy text
try:
    from src.retrieval import HybridRetriever, HashingEmbedder
    retriever = HybridRetriever.from_records(corpus, embedder=HashingEmbedder())
except Exception as e:
    print("Retriever unavailable, continuing structural-only:", e)
    retriever = None

grag = GraphRAG(graph=g, retriever=retriever)
answer = grag.ask(
    "Are there unauthenticated flows from external entities into PII data stores?"
)
print("Structural findings:")
for f in answer.structural_findings:
    print(" ", f)
print("\nPolicy hits:")
for h in answer.policy_hits:
    print(" ", h.get("control_id"), h.get("section"))
print("\nNotes:", answer.notes)

## Contrast with pure vector search

Pure vector search over policy text can retrieve SEC-DFD-014 and SEC-DFD-001, but it cannot tell you whether *this diagram* actually contains a path from Partner API to Customer PII that crosses the DMZ. The graph layer supplies that structural fact; the vector / hybrid layer supplies the governing control language.

**Submission note:** Describe how you would persist this graph in Neo4j or Spanner Graph for production (4–6 sentences).